# Overview

A run folder is one scanning probe microscopy run as the instrument exports it: `summary.json` with the recipe, the session and one record per measured point, and `loops/` with the raw curves.
This example creates the specimen on the platform as a Sample Set holding one Sample per measured position, the run as a Measurement Set holding one Measurement per Sample with the Setup it was measured on, the run's records as files on each Measurement, and one hysteresis-loop Property per Sample.
Re-running it adds only what is missing.

## Install the API client

The samples, measurements and files endpoints are not released yet, so the client is installed from its branch until it merges.

In [ ]:
%pip install -q "git+https://github.com/mat3ra/api-client.git@feature/SOF-8051"

## Authenticate and initialize API client

### Authenticate
Authenticate in the browser (OIDC device flow) or via JupyterLite host injection. Credentials are stored in environment variables.

### Initialize API client
Create an authenticated API client and resolve the owner account ID.

In [ ]:
from mat3ra.notebooks_utils.packages import install_packages

await install_packages("api")

In [ ]:
from mat3ra.notebooks_utils.auth import authenticate

await authenticate()

In [ ]:
import os

from mat3ra.api_client import APIClient

client = APIClient.authenticate()
selected_account = client.my_account
OWNER_ID = os.getenv("ORGANIZATION_ID") or selected_account.id

## Set Parameters

- **HOST**: platform the run is uploaded to
- **RUN_DIR**: path to the run folder, relative to this notebook
- **ACCOUNT_SLUG**: account the data belongs to, empty for the default account
- **FILES**: which files to upload per measurement

In [ ]:
HOST = os.environ.get("MAT3RA_HOST", "https://platform.mat3ra.com")
RUN_DIR = "20260901_171601_alscn_01448"
ACCOUNT_SLUG = ""
FILES = "records"  # "records": the record JSONs, "all": also the loop arrays and plots, "none": no files

## Fetch the uploader

Download `upload_run.py`, the parser and uploader the platform serves, into the working directory.

In [ ]:
from pathlib import Path

from mat3ra.notebooks_utils.io import read_from_url

if not Path("upload_run.py").exists():
    Path("upload_run.py").write_text(await read_from_url(f"{HOST}/upload_run.py"))

from upload_run import account_id, parse, upload

## Parse the run folder

Read the run folder into the documents the platform stores. Nothing is uploaded yet.

In [ ]:
parsed = parse(Path(RUN_DIR))
file_count = sum(len(files) for files in parsed["files"].values())
print(
    f"specimen {parsed['wafer']}: {len(parsed['samples'])} samples (ordered set) · run {parsed['run']}: "
    f"{len(parsed['measurements'])} measurements (ordered set, one per sample) · {len(parsed['records'])} records "
    f"-> {file_count} files · {len(parsed['images'])} image(s) · {len(parsed['properties'])} samples with a combined "
    f"loop · no curves: {len(parsed['skipped'])} samples"
)

## Select the account

`ACCOUNT_SLUG` re-authenticates the client against that account, so the run is read and written there.

In [ ]:
if ACCOUNT_SLUG:
    client = APIClient.authenticate(account_id=account_id(client, ACCOUNT_SLUG))

## Upload the run

Create the Sample Set and its Samples, the Measurement Set and one Measurement per Sample, the files and the loop Properties.

In [ ]:
upload(client, parsed, command="both", files=FILES)

## Print the link to the run

The run is a folder in the account's Measurements tab, named after the run.

In [ ]:
account = next(item for item in client.list_accounts() if item["_id"] == client.my_account.id)
print(f"{HOST}/{account['slug']}/measurements")

## References

- [Mat3ra REST API](https://docs.mat3ra.com/rest-api/overview/)
- [Samples and measurements in the web app](https://github.com/mat3ra/web-app/pull/2976)